# 02 — Feature Engineering

In [01_eda.ipynb](01_eda.ipynb) we established the shape of the fraud problem in this
PaySim dataset. This notebook turns those observations into a clean, model-ready
table. The guiding principle throughout is **no shortcuts that wouldn't survive
contact with a real production system**: every feature here is something we could
actually compute at the moment a transaction is submitted (or, where we deliberately
break that rule for the full-feature model, we say so explicitly).

**What this notebook does:**
1. Scopes the dataset down to the transaction types where fraud can occur
2. Removes `isFlaggedFraud` as a *feature* (keeping it only as a baseline to beat later)
3. Engineers accounting-identity "error balance" features
4. Extracts time-of-day / day features from `step`
5. Encodes the transaction `type`
6. Builds account-level behavioral features (frequency, dual-role, recency) using
   only information that would have been available *before* each transaction happened
7. Saves the result to disk for reuse in every notebook that follows


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

RAW_PATH = "online_payment_fraud.csv"
df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head()


(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## 1. Scoping to TRANSFER and CASH_OUT

The EDA notebook confirmed something important: **every single fraudulent
transaction in this dataset is either a `TRANSFER` or a `CASH_OUT`.** `PAYMENT`,
`CASH_IN`, and `DEBIT` transactions have zero recorded fraud.

That's not a modeling convenience we're inventing — it's how PaySim's fraud
simulation is constructed (fraud here is modeled as *"take control of an account,
transfer the funds out, then cash out"*). Given that, training a classifier on the
full 6.36M-row dataset would mostly teach the model to recognize "this transaction
type structurally cannot be fraud," which is a fact we already know, not a pattern
worth learning. It also dilutes the fraud rate and burns compute on rows that can
never contribute a true positive.

So we scope every downstream notebook to `TRANSFER` and `CASH_OUT` transactions only.
This is a deliberate, documented modeling decision — not silent data loss. In a real
deployment, the equivalent design choice would be routing only these transaction
types through the fraud model in the first place, and letting other types skip it
entirely (cheaper and faster for the ~85% of traffic that can't be fraudulent anyway).


In [2]:
before_rows = len(df)
before_fraud = df["isFraud"].sum()

df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].reset_index(drop=True)

print(f"Rows before scoping: {before_rows:,} (fraud cases: {before_fraud:,})")
print(f"Rows after scoping:  {len(df):,} (fraud cases: {df['isFraud'].sum():,})")
print(f"Fraud rate in scoped data: {df['isFraud'].mean():.4%}")


Rows before scoping: 6,362,620 (fraud cases: 8,213)
Rows after scoping:  2,770,409 (fraud cases: 8,213)
Fraud rate in scoped data: 0.2965%


## 2. Dropping `isFlaggedFraud` as a feature

`isFlaggedFraud` is a hard-coded rule already baked into the dataset
(`TRANSFER` amount > 200,000). We are **not deleting it from existence** — we
carry it forward as a separate reference column so notebook
[03_baseline_modeling.ipynb](03_baseline_modeling.ipynb) can measure how good
this naive rule actually is (spoiler: not very).

We do, however, remove it from the feature set any model will be trained on.
Two reasons:

- **It's leakage-adjacent.** It was constructed by someone who already knew which
  transactions look suspicious. Handing it to a model as an input feature would let
  the model partially "cheat" off a rule instead of learning generalizable patterns
  — and it would make our model's true skill impossible to separate from the rule's.
- **It's the thing we're trying to beat.** The entire point of this project is to
  show that a learned model catches meaningfully more fraud than a static
  threshold rule. If the rule is also an input feature, that comparison stops
  being meaningful.

So: saved separately as a baseline reference, dropped from the modeling table.


In [3]:
# Keep a reference table (row-aligned) purely for the naive-baseline evaluation
# in 03_baseline_modeling.ipynb. This is NEVER joined back in as a model feature.
baseline_reference = df[["step", "isFraud", "isFlaggedFraud"]].copy()

df = df.drop(columns=["isFlaggedFraud"])
print("isFlaggedFraud removed from modeling table.")
print("Reference table shape:", baseline_reference.shape)


isFlaggedFraud removed from modeling table.
Reference table shape: (2770409, 3)


## 3. Error-balance features (accounting-identity mismatches)

In a consistent ledger, the sender's new balance should equal
`old balance - amount`, and the receiver's new balance should equal
`old balance + amount`. PaySim's simulation doesn't always enforce this
perfectly for fraudulent transactions (e.g. accounts drained to exactly zero
regardless of the stated amount), which means the *size of the mismatch* is
itself a signal.

We compute:

- `errorBalanceOrig = newbalanceOrig + amount - oldbalanceOrg`
- `errorBalanceDest = oldbalanceDest + amount - newbalanceDest`

A value near zero means the transaction "balances" the way a normal one would.
A large non-zero value is exactly the kind of ledger inconsistency fraud tends
to produce.

**Important caveat we'll return to in 04:** both of these features are built from
`newbalanceOrig` / `newbalanceDest`, which are only known *after* a transaction
completes. They're valid for a full post-hoc model, but they don't belong in a
real-time, pre-authorization model — we'll rebuild a leakage-free version of that
model later.


In [4]:
df["errorBalanceOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

df[["errorBalanceOrig", "errorBalanceDest"]].describe()


,errorBalanceOrig,errorBalanceDest
count,2.770409e+06,2.770409e+06
mean,2.859850e+05,-2.864713e+04
std,8.753230e+05,5.934794e+05
min,-1.000000e-02,-7.588573e+07
25%,5.185310e+04,0.000000e+00
50%,1.435971e+05,0.000000e+00
75%,2.798912e+05,0.000000e+00
max,9.244552e+07,1.000000e+07


## 4. Time features from `step`

`step` is a simulation clock: each unit is one simulated hour, and the dataset
spans 743 steps (~31 days). Raw `step` is a useful ordering key for our
time-based split later, but as a *feature* it's better decomposed into cyclical
components a model can actually use — fraud patterns plausibly cluster by hour
of day (e.g. off-hours activity) or by day.


In [5]:
df["hour_of_day"] = df["step"] % 24
df["day"] = df["step"] // 24

df[["step", "hour_of_day", "day"]].head()


,step,hour_of_day,day
0,1,1,0
1,1,1,0
2,1,1,0
3,1,1,0
4,1,1,0


## 5. Encoding `type`

After scoping to `TRANSFER` and `CASH_OUT`, `type` is a simple binary
distinction. We one-hot encode it (`type_TRANSFER`) rather than leaving it as a
string so it can feed directly into every model in this project.


In [6]:
df["type_TRANSFER"] = (df["type"] == "TRANSFER").astype(int)
df = df.drop(columns=["type"])

df["type_TRANSFER"].value_counts()


type_TRANSFER
0    2237500
1     532909
Name: count, dtype: int64

## 6. Account-level behavioral features

Fraud detection in payments leans heavily on *account behavior*, not just the
transaction in isolation: a dormant account suddenly moving money, an account
that has only ever received money now sending a large transfer, or an account
transacting far more frequently than usual are all classic signals.

We engineer three behavioral signals for **both** the sending account
(`nameOrig`) and the receiving account (`nameDest`):

1. **Transaction frequency** — how many prior transactions (in either role)
   this account has been part of
2. **Dual-role flag** — whether this account has ever appeared in the
   *opposite* role before (a sender who has previously received money, or vice
   versa)
3. **Recency** — how much time has passed since this account was last seen in
   the data

### Why this has to be done carefully

Because we ultimately evaluate this project with a **time-based split**
(train on earlier `step`s, test on later ones, mimicking real deployment), it
would be a subtle but serious form of leakage to compute these as *global*
statistics over the whole dataset — that would let a transaction in the training
set "see" information about what an account does in the future test period.

So every feature below is computed as an **expanding, point-in-time statistic**:
for a transaction at step `s`, we only ever count activity that happened at
`step < s`. An account's very first appearance in the data gets "no prior
history" values (0 transactions, no dual-role, no last-seen time) — exactly
what a production system would know at that moment.


In [7]:
# Build a long "event log": every transaction contributes one event for the
# sending account and one for the receiving account, tagged with the role
# played and a pointer back to the original row.
events = pd.concat([
    pd.DataFrame({
        "row_id": df.index,
        "step": df["step"].values,
        "account": df["nameOrig"].values,
        "role": "orig",
    }),
    pd.DataFrame({
        "row_id": df.index,
        "step": df["step"].values,
        "account": df["nameDest"].values,
        "role": "dest",
    }),
])

print(f"Event log rows: {len(events):,} (2x transaction count, as expected)")


Event log rows: 5,540,818 (2x transaction count, as expected)


In [8]:
# --- Frequency: transactions strictly before this step, per account ---
# Count how many events each account has at each step, then take a running
# (inclusive) total per account ordered by step, and subtract the current
# step's own count to get a STRICTLY-PRIOR count.
per_step_counts = (
    events.groupby(["account", "step"]).size().rename("count_this_step").reset_index()
)
per_step_counts = per_step_counts.sort_values(["account", "step"])
per_step_counts["running_total_inclusive"] = (
    per_step_counts.groupby("account")["count_this_step"].cumsum()
)
per_step_counts["prior_count"] = (
    per_step_counts["running_total_inclusive"] - per_step_counts["count_this_step"]
)

# --- Recency: most recent DISTINCT prior step this account was seen at ---
unique_account_steps = per_step_counts[["account", "step"]].copy()
unique_account_steps["prev_step"] = unique_account_steps.groupby("account")["step"].shift(1)

per_step_counts = per_step_counts.merge(unique_account_steps, on=["account", "step"], how="left")

lookup = per_step_counts[["account", "step", "prior_count", "prev_step"]]
lookup.head()


,account,step,prior_count,prev_step
0,C1000000639,249,0,NaN
1,C1000004053,327,0,NaN
2,C1000004082,354,0,NaN
3,C1000004082,370,1,354.0
4,C1000004082,374,2,370.0


In [9]:
# --- Dual-role: has this account EVER appeared in the opposite role before this step ---
first_step_as_orig = events[events["role"] == "orig"].groupby("account")["step"].min()
first_step_as_dest = events[events["role"] == "dest"].groupby("account")["step"].min()

def attach_account_features(frame, account_col, role_prefix, opposite_first_step):
    merged = frame[[account_col, "step"]].rename(columns={account_col: "account"}).merge(
        lookup, on=["account", "step"], how="left"
    )
    prior_count = merged["prior_count"].fillna(0).astype(int).values
    prev_step = merged["prev_step"]
    time_since_last = (merged["step"].values - prev_step.values)
    time_since_last = np.where(prev_step.isna(), -1, time_since_last)  # -1 = no prior history

    opp_first = merged["account"].map(opposite_first_step)
    dual_role_prior = ((opp_first.notna()) & (opp_first.values < merged["step"].values)).astype(int)

    return pd.DataFrame({
        f"{role_prefix}_txn_count_prior": prior_count,
        f"{role_prefix}_time_since_last_txn": time_since_last,
        f"{role_prefix}_dual_role_prior": dual_role_prior,
    })

orig_features = attach_account_features(df, "nameOrig", "orig", first_step_as_dest)
dest_features = attach_account_features(df, "nameDest", "dest", first_step_as_orig)

df = pd.concat([df.reset_index(drop=True), orig_features.reset_index(drop=True),
                dest_features.reset_index(drop=True)], axis=1)

df[[
    "nameOrig", "orig_txn_count_prior", "orig_time_since_last_txn", "orig_dual_role_prior",
    "nameDest", "dest_txn_count_prior", "dest_time_since_last_txn", "dest_dual_role_prior",
]].head(10)


,nameOrig,orig_txn_count_prior,orig_time_since_last_txn,orig_dual_role_prior,nameDest,dest_txn_count_prior,dest_time_since_last_txn,dest_dual_role_prior
0,C1305486145,0,-1.0,0,C553264065,0,-1.0,0
1,C840083671,0,-1.0,0,C38997010,0,-1.0,0
2,C905080434,0,-1.0,0,C476402209,0,-1.0,0
3,C1670993182,0,-1.0,0,C1100439041,0,-1.0,0
4,C1984094095,0,-1.0,0,C932583850,0,-1.0,0
5,C768216420,0,-1.0,0,C1509514333,0,-1.0,0
6,C1570470538,0,-1.0,0,C824009085,0,-1.0,0
7,C512549200,0,-1.0,0,C248609774,0,-1.0,0
8,C2072313080,0,-1.0,0,C2001112025,0,-1.0,0
9,C1976401987,0,-1.0,0,C1937962514,0,-1.0,0


A quick sanity check: for the very first transaction any account ever makes in
this dataset, `*_txn_count_prior` should be 0, `*_time_since_last_txn` should
be -1 (our "no history" sentinel), and `*_dual_role_prior` should be 0. Let's
confirm, and look at the overall distribution of these new features.


In [10]:
print("First-ever appearances should have prior_count == 0:")
print((df["orig_txn_count_prior"] == 0).sum(), "of", len(df), "orig rows have zero prior orig-side count")

df[[
    "orig_txn_count_prior", "dest_txn_count_prior",
    "orig_time_since_last_txn", "dest_time_since_last_txn",
    "orig_dual_role_prior", "dest_dual_role_prior",
]].describe()


First-ever appearances should have prior_count == 0:
2768312 of 2770409 orig rows have zero prior orig-side count


,orig_txn_count_prior,dest_txn_count_prior,orig_time_since_last_txn,dest_time_since_last_txn,orig_dual_role_prior,dest_dual_role_prior
count,2.770409e+06,2.770409e+06,2.770409e+06,2.770409e+06,2.770409e+06,2.770409e+06
mean,1.329407e-03,5.216750e+00,-8.910832e-01,3.072901e+01,1.194769e-04,7.035062e-04
std,9.338895e-02,6.005700e+00,5.202656e+00,5.185986e+01,1.092990e-02,2.651437e-02
min,0.000000e+00,0.000000e+00,-1.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,1.000000e+00,-1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,3.000000e+00,-1.000000e+00,1.100000e+01,0.000000e+00,0.000000e+00
75%,0.000000e+00,7.000000e+00,-1.000000e+00,4.000000e+01,0.000000e+00,0.000000e+00
max,3.200000e+01,7.400000e+01,7.000000e+02,6.990000e+02,1.000000e+00,1.000000e+00


## 7. Final feature set and saving to disk

We keep `step`, `nameOrig`, and `nameDest` in the saved table even though they
won't all be fed directly into models — `step` is required for the time-based
split in every later notebook, and the account IDs are useful for
error-analysis drill-downs. They'll be dropped immediately before model
training, not before.

We save two artifacts, as CSV for portability and easy inspection outside pandas:

- **`engineered_transactions.csv`** — the full modeling table (features + `isFraud`)
- **`baseline_reference.csv`** — `step`, `isFraud`, and `isFlaggedFraud`, kept
  *only* so 03_baseline_modeling.ipynb can score the naive rule. It is never
  merged back into the feature set.


In [11]:
final_columns = [
    "step", "nameOrig", "nameDest",
    "type_TRANSFER", "amount",
    "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
    "errorBalanceOrig", "errorBalanceDest",
    "hour_of_day", "day",
    "orig_txn_count_prior", "orig_time_since_last_txn", "orig_dual_role_prior",
    "dest_txn_count_prior", "dest_time_since_last_txn", "dest_dual_role_prior",
    "isFraud",
]

df = df[final_columns]

# Downcast dtypes: the raw CSV loads everything as int64/float64, which is far
# more precision than these columns need. This roughly halves the on-disk size
# and speeds up every notebook that reads this file back in.
float_cols = [
    "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
    "errorBalanceOrig", "errorBalanceDest",
]
int32_cols = [
    "step", "hour_of_day", "day", "type_TRANSFER", "isFraud",
    "orig_txn_count_prior", "orig_time_since_last_txn", "orig_dual_role_prior",
    "dest_txn_count_prior", "dest_time_since_last_txn", "dest_dual_role_prior",
]
df[float_cols] = df[float_cols].astype("float32")
df[int32_cols] = df[int32_cols].astype("int32")

print(df.shape)
df.head()


(2770409, 20)


,step,nameOrig,nameDest,type_TRANSFER,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,errorBalanceOrig,errorBalanceDest,hour_of_day,day,orig_txn_count_prior,orig_time_since_last_txn,orig_dual_role_prior,dest_txn_count_prior,dest_time_since_last_txn,dest_dual_role_prior,isFraud
0,1,C1305486145,C553264065,1,181.000000,181.0,0.0,0.0,0.000000e+00,0.000000,1.810000e+02,1,0,0,-1,0,0,-1,0,1
1,1,C840083671,C38997010,0,181.000000,181.0,0.0,21182.0,0.000000e+00,0.000000,2.136300e+04,1,0,0,-1,0,0,-1,0,1
2,1,C905080434,C476402209,0,229133.937500,15325.0,0.0,5083.0,5.151344e+04,213808.937500,1.827035e+05,1,0,0,-1,0,0,-1,0,0
3,1,C1670993182,C1100439041,1,215310.296875,705.0,0.0,22425.0,0.000000e+00,214605.296875,2.377353e+05,1,0,0,-1,0,0,-1,0,0
4,1,C1984094095,C932583850,1,311685.875000,10835.0,0.0,6267.0,2.719173e+06,300850.875000,-2.401220e+06,1,0,0,-1,0,0,-1,0,0


In [12]:
import os
os.makedirs("data/processed", exist_ok=True)

df.to_csv("data/processed/engineered_transactions.csv", index=False)
baseline_reference.to_csv("data/processed/baseline_reference.csv", index=False)

print("Saved:")
print(" - data/processed/engineered_transactions.csv", df.shape)
print(" - data/processed/baseline_reference.csv", baseline_reference.shape)


Saved:
 - data/processed/engineered_transactions.csv (2770409, 20)
 - data/processed/baseline_reference.csv (2770409, 3)


## Summary

We now have a single, leakage-aware modeling table scoped to the transaction
types where fraud is actually possible, with:

- Accounting-identity mismatch features (`errorBalanceOrig`, `errorBalanceDest`)
- Time-of-day / day features
- A binary transaction-type encoding
- Point-in-time account behavioral features (frequency, recency, dual-role)

`isFlaggedFraud` has been set aside as a baseline reference only.

**Next:** [03_baseline_modeling.ipynb](03_baseline_modeling.ipynb) establishes a
time-based train/test split, scores the naive `isFlaggedFraud` rule as a
baseline, and trains our first real model — Logistic Regression — against it.
